# Shared preprocessing for Weber's length prediction

This notebook builds one row per specimen from the reference measurements, original photos, segmentation masks, and scale CSVs. It also prepares paired thorax crops and shared model splits.

Run the cells in order from the repository root or the `notebooks` directory. The Python kernel needs `numpy`, `pandas`, `Pillow`, `pyreadr`, `scikit-image`, and `scikit-learn`. Outputs go to `data/processed/wl`.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyreadr
from IPython.display import display
from PIL import Image, ImageOps
from skimage.measure import label, regionprops
from sklearn.model_selection import KFold, train_test_split

# Locate the repository whether the kernel starts in the root or in notebooks/.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "extract_results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the ant-morphology-prediction repository.")

DATA = ROOT / "data" / "Morphology"
IMAGES = DATA / "images"
SCALES = ROOT / "extract_results"
OUTPUT = ROOT / "data" / "processed" / "wl"
THORAX_CLASS = 2
SCALE_REVIEW_THRESHOLD = 0.10
CROP_MARGIN = 0.12
CROP_SIZE = (224, 224)
SEED = 42
TEST_FRACTION = 0.20
N_FOLDS = 5

# Reject ambiguous keys before joining records from different sources.
def require_unique(frame, key, source):
    values = frame[key]
    if values.isna().any() or values.astype(str).str.strip().eq("").any():
        raise ValueError(f"{source} contains missing or blank values in {key}.")
    duplicates = values[values.duplicated(keep=False)].unique()
    if len(duplicates):
        raise ValueError(f"{source} contains duplicate values in {key}: {duplicates[:10]}.")

def inventory(folder, extension, column):
    # Include nested folders, such as vizualize/vizualize.
    paths = sorted(folder.rglob(f"*{extension}"))
    if not paths:
        raise FileNotFoundError(f"No {extension} files found in {folder}.")
    frame = pd.DataFrame({
        "image_stem": [path.stem for path in paths],
        column: [path.relative_to(ROOT).as_posix() for path in paths],
    })
    require_unique(frame, "image_stem", folder.name)
    return frame

photos = inventory(IMAGES / "original", ".jpg", "image_path")
masks = inventory(IMAGES / "mask", ".png", "mask_path")
visualizations = inventory(IMAGES / "vizualize", ".png", "visualization_path")
scale_files = sorted(SCALES.glob("scale_results_*.csv"))
if not scale_files:
    raise FileNotFoundError(f"No scale CSV files found in {SCALES}.")
display(pd.Series({
    "photos": len(photos), "masks": len(masks),
    "visualizations": len(visualizations), "scale CSVs": len(scale_files),
}, name="count"))

## 1. Assemble the specimen table

The RData file provides the reference Weber's length (WL) in millimeters. Specimen IDs identify the reference rows. The image filename stem links each row to its photo, segmentation mask, visualization, and scale record. The inventory report counts missing matches in both directions.

In [ ]:
# The RData objects follow the same row order.
rdata = pyreadr.read_r(str(DATA / "mor_dig.RData"))
measurements = rdata["mor_dig"].reset_index(drop=True)
ids = rdata["ant"]["ant"].reset_index(drop=True)
species = rdata["spe"]["spe"].reset_index(drop=True)
if len({len(measurements), len(ids), len(species)}) != 1:
    raise ValueError("The ant, spe, and mor_dig tables must have the same number of rows.")

reference = measurements[["WL"]].rename(columns={"WL": "WL_mm"}).assign(
    specimen_id=ids.astype("string").str.strip(),
    species=species,
)
require_unique(reference, "specimen_id", "mor_dig.RData")
reference["WL_mm"] = pd.to_numeric(reference["WL_mm"], errors="coerce")
# The dataset uses the p_1 profile image for each specimen.
reference["image_stem"] = reference["specimen_id"] + "_p_1"

scales = pd.concat(
    [pd.read_csv(path, encoding="utf-8-sig") for path in scale_files], ignore_index=True
)
scales["image_stem"] = scales["image"].astype("string").str.strip().str.replace(
    r"\.[^.]+$", "", regex=True
)
require_unique(scales, "image_stem", "scale CSVs")
scales = scales.rename(columns={
    "number": "scale_number", "unit": "scale_unit", "bar_px": "bar_px_auto",
    "mm_per_px": "mm_per_px_auto", "comments": "scale_comments",
})

# Left joins preserve all reference specimens for the quality report.
table = reference.copy()
inventory_report = []
for name, source, columns in [
    ("photos", photos, ["image_path"]),
    ("masks", masks, ["mask_path"]),
    ("visualizations", visualizations, ["visualization_path"]),
    ("scales", scales, ["scale_number", "scale_unit", "bar_px_auto", "mm_per_px_auto", "scale_comments"]),
]:
    table = table.merge(source[["image_stem", *columns]], on="image_stem",
                        how="left", validate="one_to_one")
    inventory_report.append({
        "source": name,
        "source_rows": len(source),
        "unmatched_specimens": int((~reference.image_stem.isin(source.image_stem)).sum()),
        "source_rows_without_reference": int((~source.image_stem.isin(reference.image_stem)).sum()),
    })
inventory_report = pd.DataFrame(inventory_report)
display(inventory_report)
print(f"{len(table):,} reference specimens")

## 2. Check automatic scales against manual annotations

The manual conversion factor is the annotated scale bar length in millimeters divided by the distance between the bar endpoints in pixels. The automatic factor is used for feature conversion on all specimens, including the annotated subset.

A relative difference above 10% or a suspicious extraction comment flags a specimen for review. This check uses the scale bar annotations independently of the WL landmarks.

In [ ]:
annotations = pd.read_csv(DATA / "WL_digitalized.txt", sep=r"\s+", index_col=0)
annotations.index.name = "image_filename"
annotations = annotations.reset_index()
annotations["image_stem"] = annotations["image_filename"].str.replace(r"\.[^.]+$", "", regex=True)
require_unique(annotations, "image_stem", "manual annotations")
# Compute the Euclidean distance between the two scale bar endpoints.
annotations["bar_px_manual"] = np.hypot(
    annotations["X2_scale"] - annotations["X1_scale"],
    annotations["Y2_scale"] - annotations["Y1_scale"],
)
manual_valid = (
    np.isfinite(annotations["scale"]) & annotations["scale"].gt(0)
    & np.isfinite(annotations["bar_px_manual"]) & annotations["bar_px_manual"].gt(0)
)
annotations["mm_per_px_manual"] = (
    annotations["scale"] / annotations["bar_px_manual"].where(manual_valid)
)
table = table.merge(
    annotations[["image_stem", "image_filename", "bar_px_manual", "mm_per_px_manual"]],
    on="image_stem", how="left", validate="one_to_one",
)
table["has_manual_annotation"] = table.pop("image_filename").notna()
# Keep the raw automatic value and use only finite, positive factors for conversion.
auto = pd.to_numeric(table["mm_per_px_auto"], errors="coerce")
table["scale_valid"] = np.isfinite(auto) & auto.gt(0)
table["mm_per_px"] = auto.where(table["scale_valid"])
table["scale_relative_error"] = (table["mm_per_px"] / table["mm_per_px_manual"] - 1).abs()
table["scale_disagreement"] = table["scale_relative_error"].gt(SCALE_REVIEW_THRESHOLD)
table["scale_ocr_review"] = table["scale_comments"].str.contains(
    r"assum|correct|please check|\bweak\b", case=False, regex=True, na=False
)
table["target_valid"] = np.isfinite(table["WL_mm"]) & table["WL_mm"].gt(0)

comparison = table.loc[table["has_manual_annotation"], [
    "specimen_id", "bar_px_auto", "bar_px_manual", "mm_per_px",
    "mm_per_px_manual", "scale_relative_error", "scale_disagreement", "scale_comments",
]].copy()
print(f"Annotations read: {len(annotations)} | matched: {len(comparison)}")
print(f"Finite, positive automatic scale factors: {table['scale_valid'].sum()}")
print(f"Valid comparisons: {comparison['scale_relative_error'].count()}")
print(f"Median relative error: {comparison['scale_relative_error'].median():.2%}")
print(f"Relative errors above {SCALE_REVIEW_THRESHOLD:.0%}: {table['scale_disagreement'].sum()}")
display(comparison.nlargest(10, "scale_relative_error"))

## 3. Measure thorax features and prepare image inputs

Measurements use the thorax mask at its original resolution. The projected length is the full span of pixel centers along the main axis. The robust version uses the span between the 1st and 99th percentiles. Scale factors convert lengths to mm and areas to mm². Ratios are dimensionless.

The mask isolates the thorax in the photo. Both inputs use the same crop, then retain their aspect ratio when centered in a 224 × 224 canvas. The specimen table stores physical dimensions measured before resizing.

[Region measurements in scikit-image](https://scikit-image.org/docs/stable/api/skimage.measure.html) · [Image padding in Pillow](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html#PIL.ImageOps.pad)

In [ ]:
visual_dir = OUTPUT / "visual"
visual_dir.mkdir(parents=True, exist_ok=True)

def prepare_specimen(row):
    if pd.isna(row.image_path) or pd.isna(row.mask_path):
        raise ValueError("missing photo or mask")

    with Image.open(ROOT / row.mask_path) as mask_image:
        labels = np.array(mask_image)
    if labels.ndim != 2:
        raise ValueError("the mask must contain a single layer of class labels")

    thorax = labels == THORAX_CLASS
    ys, xs = np.nonzero(thorax)
    if not len(ys):
        raise ValueError("the thorax class is absent from the mask")

    top, bottom = int(ys.min()), int(ys.max()) + 1
    left, right = int(xs.min()), int(xs.max()) + 1
    height, width = bottom - top, right - left
    tight_mask = thorax[top:bottom, left:right]
    # Measure all thorax pixels as one region, including disconnected fragments.
    props = regionprops(tight_mask.astype(np.uint8))[0]
    # Component sizes provide a separate check for segmentation fragmentation.
    component_sizes = np.bincount(label(tight_mask, connectivity=2).ravel())[1:]
    # The eigenvector with the largest eigenvalue gives the main axis of the silhouette.
    pixels = np.column_stack((ys, xs)).astype(float)
    centered = pixels - pixels.mean(axis=0)
    _, axes = np.linalg.eigh(centered.T @ centered)
    projection = centered @ axes[:, -1]
    lower, upper = np.percentile(projection, [1, 99])

    # Add a margin and clip the shared crop to the original image bounds.
    pad_y, pad_x = int(np.ceil(height * CROP_MARGIN)), int(np.ceil(width * CROP_MARGIN))
    crop_box = (
        max(0, left - pad_x), max(0, top - pad_y),
        min(labels.shape[1], right + pad_x), min(labels.shape[0], bottom + pad_y),
    )
    x0, y0, x1, y1 = crop_box
    crop_mask = thorax[y0:y1, x0:x1]
    with Image.open(ROOT / row.image_path) as source:
        if source.size != (labels.shape[1], labels.shape[0]):
            raise ValueError(f"size mismatch between the photo {source.size} and mask {labels.shape[::-1]}")
        crop_rgb = np.array(source.crop(crop_box).convert("RGB"))

    # Remove background pixels and use nearest-neighbor interpolation for binary labels.
    crop_rgb[~crop_mask] = 255
    photo = ImageOps.pad(Image.fromarray(crop_rgb), CROP_SIZE, Image.Resampling.LANCZOS, color="white")
    mask = ImageOps.pad(Image.fromarray(crop_mask.astype(np.uint8) * 255), CROP_SIZE,
                        Image.Resampling.NEAREST, color=0)
    photo_path = visual_dir / f"{row.image_stem}_photo.png"
    mask_path = visual_dir / f"{row.image_stem}_mask.png"
    photo.save(photo_path)
    mask.save(mask_path)

    return {
        "thorax_width_px": width,
        "thorax_height_px": height,
        "area_px": props.area,
        "perimeter_px": props.perimeter,
        "major_axis_px": props.major_axis_length,
        "minor_axis_px": props.minor_axis_length,
        "feret_max_px": props.feret_diameter_max,
        "projected_length_px": np.ptp(projection),
        "projected_p01_p99_px": upper - lower,
        "minor_major_ratio": props.minor_axis_length / props.major_axis_length if props.major_axis_length else np.nan,
        "aspect_ratio": width / height,
        "eccentricity": props.eccentricity,
        "solidity": props.solidity,
        "extent": props.extent,
        "component_count": len(component_sizes),
        "largest_component_ratio": component_sizes.max() / component_sizes.sum(),
        "crop_left": x0, "crop_top": y0, "crop_right": x1, "crop_bottom": y1,
        "thorax_photo_path": photo_path.relative_to(ROOT).as_posix(),
        "thorax_mask_path": mask_path.relative_to(ROOT).as_posix(),
    }

In [ ]:
records = []
for index, row in enumerate(table.itertuples(index=False), start=1):
    record = {"specimen_id": row.specimen_id, "visual_issue": None}
    try:
        record.update(prepare_specimen(row))
    except (OSError, ValueError) as error:
        record["visual_issue"] = str(error)
    records.append(record)
    if index % 500 == 0:
        print(f"{index}/{len(table)} specimens processed", flush=True)

features = pd.DataFrame.from_records(records)
if features["visual_issue"].notna().all():
    display(features.head())
    raise RuntimeError("No photo and mask pairs could be processed.")

table = table.merge(features, on="specimen_id", validate="one_to_one")
# Apply the linear scale to distances and its square to areas.
length_columns = [
    "thorax_width_px", "thorax_height_px", "perimeter_px", "major_axis_px", "minor_axis_px",
    "feret_max_px", "projected_length_px", "projected_p01_p99_px",
]
for column in length_columns:
    table[column.removesuffix("_px") + "_mm"] = table[column] * table["mm_per_px"]
table["area_mm2"] = table["area_px"] * table["mm_per_px"].pow(2)
# Square-root area provides a measure of size in millimeters for the LF feature set.
table["sqrt_area_mm"] = np.sqrt(table["area_mm2"])
# Flag masks with a small largest component or low convex-hull coverage.
table["mask_review"] = table["largest_component_ratio"].lt(0.95) | table["solidity"].lt(0.5)
table["visual_ready"] = table["visual_issue"].isna()
print(f"Cropped pairs: {table['visual_ready'].sum()}/{len(table)}")

sample = table.loc[table["visual_ready"]].iloc[0]
with Image.open(ROOT / sample["thorax_photo_path"]) as photo, Image.open(ROOT / sample["thorax_mask_path"]) as mask:
    preview = Image.new("RGB", (2 * CROP_SIZE[0], CROP_SIZE[1]), "white")
    preview.paste(photo, (0, 0))
    preview.paste(mask.convert("RGB"), (CROP_SIZE[0], 0))
display(preview)

## 4. Select eligible specimens and create shared splits

All models use specimens with a finite, positive WL and scale factor, both cropped inputs, and a complete set of tabular features. Specimens flagged for review remain eligible when they meet these requirements. The review file records the reasons for flags and exclusions.

The test set holds 20% of eligible specimens. The remaining specimen IDs are sorted, then assigned to five cross-validation folds with seed 42. For each fold in `partitions.csv`, use development rows with that fold number for validation and the other development rows for training. Fit any feature standardization within the training rows of each fold.

In [ ]:
length_features = [
    "feret_max_mm", "major_axis_mm", "thorax_width_mm", "projected_length_mm", "minor_axis_mm",
]
shape_features = length_features + [
    "sqrt_area_mm", "minor_major_ratio", "solidity", "extent", "perimeter_mm",
]
table["features_complete"] = np.isfinite(table[shape_features]).all(axis=1)
required_checks = {
    "target_valid": "missing or invalid WL",
    "scale_valid": "missing or invalid automatic scale",
    "visual_ready": "cropped photo or mask unavailable",
    "features_complete": "incomplete features",
}
review_checks = {
    "scale_disagreement": "automatic scale differs from manual scale",
    "scale_ocr_review": "OCR comment requires review",
    "mask_review": "fragmented mask or low solidity",
}
# Review flags are informational. Only failed required checks affect eligibility.
table["eligible"] = table[list(required_checks)].all(axis=1)
table["needs_review"] = table[list(review_checks)].any(axis=1)

def describe_flags(flags, reasons):
    return flags.apply(lambda row: " | ".join(reasons[key] for key, active in row.items() if active), axis=1)

table["exclusion_reasons"] = describe_flags(~table[list(required_checks)], required_checks)
table["review_reasons"] = describe_flags(table[list(review_checks)], review_checks)

# Sort unique specimen IDs before splitting to make assignments reproducible.
eligible_ids = sorted(table.loc[table["eligible"], "specimen_id"])
dev_ids, test_ids = train_test_split(eligible_ids, test_size=TEST_FRACTION, random_state=SEED)
dev_ids = sorted(dev_ids)
partitions = pd.DataFrame({"specimen_id": dev_ids, "split": "development"})
partitions["validation_fold"] = pd.Series(pd.NA, index=partitions.index, dtype="Int64")
# Store each development specimen's validation fold once.
for fold, (_, val_indices) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(dev_ids), start=1):
    partitions.loc[val_indices, "validation_fold"] = fold
partitions = pd.concat([
    partitions,
    pd.DataFrame({
        "specimen_id": sorted(test_ids), "split": "test",
        "validation_fold": pd.Series(pd.NA, index=range(len(test_ids)), dtype="Int64"),
    }),
], ignore_index=True)
table = table.merge(partitions, on="specimen_id", how="left", validate="one_to_one")
display(partitions.groupby(["split", "validation_fold"], dropna=False).size().rename("specimen_count"))

## 5. Export the shared files

`dataset_preprocessing.csv` contains every specimen, its reference WL, its measurements, repository-relative paths, and quality indicators. `feature_sets.json` lists the inputs for the length-only set (L) and the length-plus-shape set (LF).

Use `partitions.csv` to reproduce the splits, `scale_comparison.csv` to review the manual scale comparison, and `review_cases.csv` to inspect flagged or excluded specimens. `inventory.csv` summarizes the input matches.

In [ ]:
table.to_csv(OUTPUT / "dataset_preprocessing.csv", index=False)
partitions.to_csv(OUTPUT / "partitions.csv", index=False)
comparison.to_csv(OUTPUT / "scale_comparison.csv", index=False)
inventory_report.to_csv(OUTPUT / "inventory.csv", index=False)
review_columns = [
    "specimen_id", "eligible", "needs_review", "exclusion_reasons", "review_reasons",
    "mm_per_px", "mm_per_px_manual", "scale_relative_error", "scale_comments",
    "solidity", "largest_component_ratio", "visual_issue",
]
table.loc[~table["eligible"] | table["needs_review"], review_columns].to_csv(
    OUTPUT / "review_cases.csv", index=False
)
# Export explicit model inputs to keep target and quality columns out of the feature matrix.
(OUTPUT / "feature_sets.json").write_text(
    json.dumps({"target": "WL_mm", "L": length_features, "LF": shape_features}, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Table rows: {len(table)} | eligible specimens: {table['eligible'].sum()} | test specimens: {len(test_ids)}")
print(f"Flagged specimens: {table['needs_review'].sum()} | excluded: {(~table['eligible']).sum()}")
print(f"Output directory: {OUTPUT.relative_to(ROOT)}")